# Curadoria de Datasets Acadêmicos — FakeTrueBR

**Versão:** 1  
**Criado em:** 2026-05-24  

## Objetivo
Curar o corpus FakeTrueBR padronizado (Fase 2), gerando dois arquivos curated em  
`dados/pipeline_datasets_academicos/curated/faketruebr/`.

## Constraints obrigatórias
- `dataset_final_treino_v1.csv` e `dataset_final_treino_v2*.csv` **não são lidos nem alterados**
- Nenhum modelo é treinado
- Nenhum arquivo curated existente é sobrescrito — todos os outputs têm timestamp
- O arquivo `faketruebr_raw_padronizado_*.csv` **não é modificado**
- `dataset_final_treino_v3` **não é criado neste notebook**
- Rastreabilidade preservada: `id_registro`, `fonte_dataset`, `referencia_dataset`, `url_origem`, `portal_origem`, `origem_qualidade`

## Saídas geradas
| Arquivo | Conteúdo |
|---|---|
| `faketruebr_curated_full_YYYY-MM-DD_HH-MM-SS.csv` | Todos os registros limpos e deduplicados |
| `faketruebr_curated_size_control_YYYY-MM-DD_HH-MM-SS.csv` | Versão com controle de viés de tamanho |
| `faketruebr_conflitos_YYYY-MM-DD_HH-MM-SS.csv` | Registros com conflito de label (se houver) |

## Etapas
1. Configuração e localização automática do raw mais recente  
2. Leitura e diagnóstico inicial  
3. Limpeza básica  
4. Deduplicação  
5. Conflito de labels  
6. Auditoria do viés de tamanho  
7. Geração dos arquivos curated  
8. Relatório final

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
from pathlib import Path


def _find_project_root() -> Path:
    cwd = Path.cwd()
    if (cwd / 'dados').is_dir():
        return cwd
    if (cwd.parent / 'dados').is_dir():
        return cwd.parent
    raise FileNotFoundError(
        f'Pasta dados nao encontrada em {Path.cwd()}. '
        'Execute o notebook a partir da raiz do projeto ou de src/.'
    )


PROJECT_ROOT = _find_project_root()

# --- Caminhos ---
PIPELINE_DIR  = PROJECT_ROOT / 'dados' / 'pipeline_datasets_academicos'
RAW_DIR       = PIPELINE_DIR / 'raw' / 'faketruebr'
CURATED_DIR   = PIPELINE_DIR / 'curated' / 'faketruebr'
CURATED_DIR.mkdir(parents=True, exist_ok=True)

# --- Localizar o raw padronizado mais recente ---
padronizados = sorted(RAW_DIR.glob('faketruebr_raw_padronizado_*.csv'))
assert padronizados, f'Nenhum arquivo faketruebr_raw_padronizado_*.csv em {RAW_DIR}'
ARQUIVO_RAW = padronizados[-1]

# --- Schema obrigatorio ---
COLUNAS_SCHEMA = [
    'id_registro', 'texto_principal', 'label', 'label_detalhe',
    'pipeline_origem', 'portal_origem', 'origem_texto', 'origem_qualidade',
    'tamanho_chars', 'data_publicacao', 'url_origem',
    'fonte_dataset', 'referencia_dataset',
]

# --- Timestamp para todos os outputs ---
TS = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')

SEP = '=' * 65

print(f'PROJECT_ROOT : {PROJECT_ROOT}')
print(f'RAW_DIR      : {RAW_DIR}')
print(f'CURATED_DIR  : {CURATED_DIR}')
print(f'ARQUIVO_RAW  : {ARQUIVO_RAW.name}')
print(f'Timestamp    : {TS}')

PROJECT_ROOT : C:\Users\offan\Desktop\ml-checkai
RAW_DIR      : C:\Users\offan\Desktop\ml-checkai\dados\pipeline_datasets_academicos\raw\faketruebr
CURATED_DIR  : C:\Users\offan\Desktop\ml-checkai\dados\pipeline_datasets_academicos\curated\faketruebr
ARQUIVO_RAW  : faketruebr_raw_padronizado_2026-05-24_19-57-33.csv
Timestamp    : 2026-05-24_20-01-57


## Seção 1 — Leitura e diagnóstico inicial

Lê o arquivo raw padronizado mais recente e exibe estatísticas completas antes de qualquer curadoria.

In [2]:
df_raw = pd.read_csv(ARQUIVO_RAW, encoding='utf-8')

print(SEP)
print('SECAO 1 — LEITURA E DIAGNOSTICO INICIAL')
print(SEP)
print(f'Arquivo: {ARQUIVO_RAW.name}')
print(f'Shape  : {df_raw.shape}')
print()

# Validar colunas obrigatórias do schema CheckAI
colunas_faltando = [c for c in COLUNAS_SCHEMA if c not in df_raw.columns]
if colunas_faltando:
    print(f'ATENCAO — Colunas faltando no schema: {colunas_faltando}')
else:
    print('Schema CheckAI OK — todas as colunas obrigatorias presentes.')
print()

# Distribuição por label
print('--- Distribuicao por label ---')
dist_label = df_raw['label'].value_counts().sort_index()
for lbl, cnt in dist_label.items():
    nome = 'fake' if lbl == 0 else 'real'
    print(f'  label={lbl} ({nome}): {cnt}')
print()

# Distribuição por label_detalhe
print('--- Distribuicao por label_detalhe ---')
print(df_raw['label_detalhe'].value_counts().to_string())
print()

# Estatísticas de tamanho por label
print('--- Estatisticas de tamanho_chars por label ---')
for lbl in [0, 1]:
    s = df_raw[df_raw['label'] == lbl]['tamanho_chars']
    nome = 'fake' if lbl == 0 else 'real'
    print(f'label={lbl} ({nome}):')
    print(f'  media  : {s.mean():.0f}')
    print(f'  mediana: {s.median():.0f}')
    print(f'  min    : {s.min()}')
    print(f'  max    : {s.max()}')
print()

# Percentis de tamanho por label
print('--- Percentis de tamanho_chars por label ---')
percentis = [10, 25, 50, 75, 90, 95]
for lbl in [0, 1]:
    s = df_raw[df_raw['label'] == lbl]['tamanho_chars']
    nome = 'fake' if lbl == 0 else 'real'
    print(f'label={lbl} ({nome}):')
    for p in percentis:
        print(f'  P{p:2d}: {s.quantile(p/100):.0f}')
print()

# Distribuição por portal_origem
print('--- Distribuicao por portal_origem ---')
print(df_raw['portal_origem'].value_counts().to_string())
print()

# Exemplos de textos muito curtos
print('--- Exemplos de textos MUITO CURTOS (< 200 chars) ---')
curtos = df_raw[df_raw['tamanho_chars'] < 200]
if curtos.empty:
    print('  (nenhum < 200 chars)')
else:
    for _, row in curtos.head(3).iterrows():
        print(f"  [{row['label_detalhe']}] ({row['tamanho_chars']} chars): {str(row['texto_principal'])[:150]}")
print()

# Exemplos de textos muito longos
print('--- Exemplos de textos MUITO LONGOS (> 10000 chars) ---')
longos = df_raw[df_raw['tamanho_chars'] > 10000]
if longos.empty:
    print('  (nenhum > 10000 chars)')
else:
    for _, row in longos.head(3).iterrows():
        print(f"  [{row['label_detalhe']}] ({row['tamanho_chars']} chars): {str(row['texto_principal'])[:150]}...")

SECAO 1 — LEITURA E DIAGNOSTICO INICIAL
Arquivo: faketruebr_raw_padronizado_2026-05-24_19-57-33.csv
Shape  : (3582, 13)

Schema CheckAI OK — todas as colunas obrigatorias presentes.

--- Distribuicao por label ---
  label=0 (fake): 1791
  label=1 (real): 1791

--- Distribuicao por label_detalhe ---
label_detalhe
FAKETRUEBR_FAKE    1791
FAKETRUEBR_REAL    1791

--- Estatisticas de tamanho_chars por label ---
label=0 (fake):
  media  : 859
  mediana: 650
  min    : 121
  max    : 6973
label=1 (real):
  media  : 3024
  mediana: 2223
  min    : 298
  max    : 32766

--- Percentis de tamanho_chars por label ---
label=0 (fake):
  P10: 285
  P25: 400
  P50: 650
  P75: 1068
  P90: 1689
  P95: 2184
label=1 (real):
  P10: 1065
  P25: 1481
  P50: 2223
  P75: 3607
  P90: 5550
  P95: 7566

--- Distribuicao por portal_origem ---
portal_origem
Boatos.org          1791
G1 Globo            1533
Desconhecido         155
Folha de S.Paulo     103

--- Exemplos de textos MUITO CURTOS (< 200 chars) ---
  [F

## Seção 2 — Limpeza básica

Operações:
1. Remover `texto_principal` vazio ou nulo
2. Remover linhas sem `label` válido
3. Remover espaços duplicados e quebras de linha excessivas
4. Recalcular `tamanho_chars`
5. Criar coluna `faixa_tamanho` (curto / medio / longo / muito_longo)

In [3]:
print(SEP)
print('SECAO 2 — LIMPEZA BASICA')
print(SEP)

df_clean = df_raw.copy()
n_inicial = len(df_clean)

# 1. Remover texto_principal vazio ou nulo
df_clean['texto_principal'] = df_clean['texto_principal'].astype(str)
mask_vazio = (
    df_clean['texto_principal'].isin(['', 'nan', 'None', 'NaN'])
    | df_clean['texto_principal'].isna()
)
n_sem_texto = int(mask_vazio.sum())
df_clean = df_clean[~mask_vazio].copy()

# 2. Remover linhas sem label válido
mask_sem_label = ~df_clean['label'].isin([0, 1])
n_sem_label = int(mask_sem_label.sum())
df_clean = df_clean[~mask_sem_label].copy()

# 3. Normalizar espaços e quebras de linha
df_clean['texto_principal'] = (
    df_clean['texto_principal']
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

# 4. Recalcular tamanho_chars
df_clean['tamanho_chars'] = df_clean['texto_principal'].str.len()

# 5. Criar faixa_tamanho
def classificar_faixa(n):
    if n < 500:
        return 'curto'
    elif n < 2000:
        return 'medio'
    elif n < 5000:
        return 'longo'
    else:
        return 'muito_longo'

df_clean['faixa_tamanho'] = df_clean['tamanho_chars'].apply(classificar_faixa)

n_apos_limpeza = len(df_clean)

print(f'Registros iniciais       : {n_inicial}')
print(f'  Removidos (sem texto)  : {n_sem_texto}')
print(f'  Removidos (sem label)  : {n_sem_label}')
print(f'Registros apos limpeza   : {n_apos_limpeza}')
print()
print('--- Distribuicao por label apos limpeza ---')
for lbl in [0, 1]:
    cnt = (df_clean['label'] == lbl).sum()
    nome = 'fake' if lbl == 0 else 'real'
    print(f'  label={lbl} ({nome}): {cnt}')
print()
print('--- Tabela cruzada: faixa_tamanho x label (apos limpeza) ---')
ordem_faixas = ['curto', 'medio', 'longo', 'muito_longo']
tab_faixa = pd.crosstab(df_clean['faixa_tamanho'], df_clean['label'], margins=True)
tab_faixa.columns = ['label=0 (fake)', 'label=1 (real)', 'Total']
idx_existente = [f for f in ordem_faixas if f in tab_faixa.index] + ['All']
tab_faixa = tab_faixa.reindex(idx_existente)
print(tab_faixa.to_string())

SECAO 2 — LIMPEZA BASICA


Registros iniciais       : 3582
  Removidos (sem texto)  : 0
  Removidos (sem label)  : 0
Registros apos limpeza   : 3582

--- Distribuicao por label apos limpeza ---
  label=0 (fake): 1791
  label=1 (real): 1791

--- Tabela cruzada: faixa_tamanho x label (apos limpeza) ---
               label=0 (fake)  label=1 (real)  Total
faixa_tamanho                                       
curto                     634              34    668
medio                    1042             748   1790
longo                     112             781    893
muito_longo                 3             228    231
All                      1791            1791   3582


## Seção 3 — Deduplicação

Etapas:
1. Criar `texto_norm` (lowercase + strip + espaços normalizados) para deduplicação interna
2. Remover duplicatas exatas por `texto_norm`, mantendo a primeira ocorrência
3. Remover duplicatas por `url_origem` (apenas URLs não vazias), mantendo a primeira ocorrência

In [4]:
print(SEP)
print('SECAO 3 — DEDUPLICACAO')
print(SEP)

df_dedup = df_clean.copy()
n_antes_dedup = len(df_dedup)

# Criar texto_norm para deduplicação
df_dedup['texto_norm'] = (
    df_dedup['texto_principal']
    .str.lower()
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
)

# 1. Deduplicar por texto_norm
n_dup_texto = int(df_dedup.duplicated(subset=['texto_norm']).sum())
df_dedup = df_dedup.drop_duplicates(subset=['texto_norm'], keep='first').copy()
n_removidos_texto = n_antes_dedup - len(df_dedup)

# 2. Deduplicar por url_origem (somente URLs não vazias)
mask_tem_url = df_dedup['url_origem'].fillna('').str.strip() != ''
n_dup_url = int(df_dedup[mask_tem_url].duplicated(subset=['url_origem']).sum())

idx_url_ok = df_dedup[mask_tem_url].drop_duplicates(subset=['url_origem'], keep='first').index
idx_sem_url = df_dedup[~mask_tem_url].index
df_dedup = df_dedup.loc[sorted(idx_url_ok.union(idx_sem_url))].copy()
n_removidos_url = n_antes_dedup - n_removidos_texto - len(df_dedup)

n_apos_dedup = len(df_dedup)

print(f'Registros antes dedup      : {n_antes_dedup}')
print(f'  Dup por texto_norm       : {n_dup_texto} → removidos: {n_removidos_texto}')
print(f'  Dup por url_origem       : {n_dup_url} → removidos: {n_removidos_url}')
print(f'Registros apos dedup       : {n_apos_dedup}')
print()
print('--- Distribuicao por label apos dedup ---')
for lbl in [0, 1]:
    cnt = (df_dedup['label'] == lbl).sum()
    nome = 'fake' if lbl == 0 else 'real'
    print(f'  label={lbl} ({nome}): {cnt}')

SECAO 3 — DEDUPLICACAO


Registros antes dedup      : 3582
  Dup por texto_norm       : 400 → removidos: 400
  Dup por url_origem       : 87 → removidos: 87
Registros apos dedup       : 3095

--- Distribuicao por label apos dedup ---
  label=0 (fake): 1752
  label=1 (real): 1343


## Seção 4 — Conflito de labels

Verificar se o mesmo texto ou mesma URL aparece com labels diferentes.  
Registros com conflito são removidos do dataset curado e salvos separadamente.

In [5]:
print(SEP)
print('SECAO 4 — CONFLITO DE LABELS')
print(SEP)

# Conflitos por texto_norm
dup_texto = (
    df_dedup.groupby('texto_norm')['label']
    .nunique()
    .reset_index(name='n_labels')
)
textos_conflito = set(dup_texto[dup_texto['n_labels'] > 1]['texto_norm'])
n_conflito_texto = len(textos_conflito)

# Conflitos por url_origem
df_url_validas = df_dedup[df_dedup['url_origem'].fillna('').str.strip() != ''].copy()
dup_url = (
    df_url_validas.groupby('url_origem')['label']
    .nunique()
    .reset_index(name='n_labels')
)
urls_conflito = set(dup_url[dup_url['n_labels'] > 1]['url_origem'])
n_conflito_url = len(urls_conflito)

# Separar conflitos do dataset limpo
mask_conflito = (
    df_dedup['texto_norm'].isin(textos_conflito)
    | df_dedup['url_origem'].isin(urls_conflito)
)
df_conflitos = df_dedup[mask_conflito].copy()
df_sem_conflito = df_dedup[~mask_conflito].copy()

n_conflitos_total = len(df_conflitos)
n_apos_conflito = len(df_sem_conflito)

ARQUIVO_CONFLITOS = None

print(f'Conflitos por texto_norm  : {n_conflito_texto} texto(s)')
print(f'Conflitos por url_origem  : {n_conflito_url} URL(s)')
print(f'Registros afetados        : {n_conflitos_total}')
print(f'Registros limpos          : {n_apos_conflito}')
print()

if n_conflitos_total > 0:
    ARQUIVO_CONFLITOS = CURATED_DIR / f'faketruebr_conflitos_{TS}.csv'
    cols_conflito = [c for c in df_conflitos.columns if c != 'texto_norm']
    df_conflitos[cols_conflito].to_csv(ARQUIVO_CONFLITOS, index=False, encoding='utf-8')
    print(f'Conflitos salvos: {ARQUIVO_CONFLITOS.name}')
else:
    print('Nenhum conflito de label detectado.')

print()
print('--- Distribuicao por label apos remocao de conflitos ---')
for lbl in [0, 1]:
    cnt = (df_sem_conflito['label'] == lbl).sum()
    nome = 'fake' if lbl == 0 else 'real'
    print(f'  label={lbl} ({nome}): {cnt}')

SECAO 4 — CONFLITO DE LABELS
Conflitos por texto_norm  : 0 texto(s)
Conflitos por url_origem  : 0 URL(s)
Registros afetados        : 0
Registros limpos          : 3095

Nenhum conflito de label detectado.

--- Distribuicao por label apos remocao de conflitos ---
  label=0 (fake): 1752
  label=1 (real): 1343


## Seção 5 — Auditoria do viés de tamanho

Compara `tamanho_chars` entre `label=0` e `label=1` após toda a limpeza.  
**Critério:** viés é considerado *relevante* se a mediana de uma classe for ≥ 2× a da outra.

In [6]:
print(SEP)
print('SECAO 5 — AUDITORIA DO VIES DE TAMANHO')
print(SEP)

df_audit = df_sem_conflito.copy()

medias    = {}
medianas  = {}
for lbl in [0, 1]:
    s = df_audit[df_audit['label'] == lbl]['tamanho_chars']
    medias[lbl]   = s.mean()
    medianas[lbl] = s.median()

print('--- Estatisticas de tamanho por label ---')
for lbl in [0, 1]:
    s = df_audit[df_audit['label'] == lbl]['tamanho_chars']
    nome = 'fake' if lbl == 0 else 'real'
    print(f'label={lbl} ({nome}):')
    print(f'  media  : {medias[lbl]:.0f}')
    print(f'  mediana: {medianas[lbl]:.0f}')
    print(f'  min    : {s.min()}')
    print(f'  max    : {s.max()}')
print()

ratio_mediana = max(medianas[0], medianas[1]) / min(medianas[0], medianas[1])
ratio_media   = max(medias[0], medias[1]) / min(medias[0], medias[1])
print(f'Ratio medianas (label=1 / label=0): {ratio_mediana:.2f}x')
print(f'Ratio medias   (label=1 / label=0): {ratio_media:.2f}x')
print()

# Tabela cruzada label x faixa_tamanho
print('--- Tabela cruzada: label x faixa_tamanho ---')
ordem_faixas = ['curto', 'medio', 'longo', 'muito_longo']
tab_cruzada = pd.crosstab(df_audit['faixa_tamanho'], df_audit['label'], margins=True)
tab_cruzada.columns = ['label=0 (fake)', 'label=1 (real)', 'Total']
tab_cruzada.index.name = 'faixa_tamanho'
idx_existente = [f for f in ordem_faixas if f in tab_cruzada.index] + ['All']
tab_cruzada = tab_cruzada.reindex(idx_existente)
print(tab_cruzada.to_string())
print()

# Diagnóstico textual
print('--- Diagnostico de vies ---')
VIES_RELEVANTE = ratio_mediana >= 2.0
if VIES_RELEVANTE:
    print(f'[VIES RELEVANTE] Ratio de medianas = {ratio_mediana:.2f}x (>= 2.0 — limite metodologico).')
    print(f'  Mediana label=1 ({medianas[1]:.0f} chars) e {ratio_mediana:.2f}x maior que label=0 ({medianas[0]:.0f} chars).')
    print('  Recomendacao: usar curated_size_control como ablacao.')
    print('  curated_full pode ser usado com class_weight para compensar.')
else:
    print(f'[VIES MODERADO] Ratio de medianas = {ratio_mediana:.2f}x (< 2.0).')
    print('  curated_full pode ser usado diretamente sem ajuste de tamanho.')

SECAO 5 — AUDITORIA DO VIES DE TAMANHO
--- Estatisticas de tamanho por label ---
label=0 (fake):
  media  : 861
  mediana: 654
  min    : 121
  max    : 6973
label=1 (real):
  media  : 3080
  mediana: 2198
  min    : 298
  max    : 32765

Ratio medianas (label=1 / label=0): 3.36x
Ratio medias   (label=1 / label=0): 3.58x

--- Tabela cruzada: label x faixa_tamanho ---
               label=0 (fake)  label=1 (real)  Total
faixa_tamanho                                       
curto                     618              31    649
medio                    1021             563   1584
longo                     110             567    677
muito_longo                 3             182    185
All                      1752            1343   3095

--- Diagnostico de vies ---
[VIES RELEVANTE] Ratio de medianas = 3.36x (>= 2.0 — limite metodologico).
  Mediana label=1 (2198 chars) e 3.36x maior que label=0 (654 chars).
  Recomendacao: usar curated_size_control como ablacao.
  curated_full pode ser usado

## Seção 6 — Geração dos arquivos curated

### 6.1 — faketruebr_curated_full
Contém todos os registros limpos, deduplicados e sem conflito.  
Mantém textos longos. Uso futuro: análise e treino com `class_weight`.

### 6.2 — faketruebr_curated_size_control
Versão com controle de viés de tamanho.  
Estratégia: filtrar para `tamanho_chars` entre 300 e 3000, depois balancear por label.

In [7]:
print(SEP)
print('SECAO 6.1 — GERANDO faketruebr_curated_full')
print(SEP)

COLUNAS_CURATED = COLUNAS_SCHEMA + ['faixa_tamanho']
df_curated_full = df_sem_conflito[COLUNAS_CURATED].copy().reset_index(drop=True)

ARQUIVO_FULL = CURATED_DIR / f'faketruebr_curated_full_{TS}.csv'
df_curated_full.to_csv(ARQUIVO_FULL, index=False, encoding='utf-8')

print(f'Total registros : {len(df_curated_full)}')
for lbl in [0, 1]:
    s = df_curated_full[df_curated_full['label'] == lbl]
    nome = 'fake' if lbl == 0 else 'real'
    print(f'  label={lbl} ({nome}): {len(s)} | media={s["tamanho_chars"].mean():.0f} | mediana={s["tamanho_chars"].median():.0f}')
print()
print(f'Arquivo salvo: {ARQUIVO_FULL.name}')
print(f'Caminho      : {ARQUIVO_FULL}')

SECAO 6.1 — GERANDO faketruebr_curated_full
Total registros : 3095
  label=0 (fake): 1752 | media=861 | mediana=654
  label=1 (real): 1343 | media=3080 | mediana=2198

Arquivo salvo: faketruebr_curated_full_2026-05-24_20-01-57.csv
Caminho      : C:\Users\offan\Desktop\ml-checkai\dados\pipeline_datasets_academicos\curated\faketruebr\faketruebr_curated_full_2026-05-24_20-01-57.csv


In [8]:
print(SEP)
print('SECAO 6.2 — GERANDO faketruebr_curated_size_control')
print(SEP)

MINIMO_CHARS = 300
MAXIMO_CHARS = 3000

df_filtrado = df_sem_conflito[
    (df_sem_conflito['tamanho_chars'] >= MINIMO_CHARS) &
    (df_sem_conflito['tamanho_chars'] <= MAXIMO_CHARS)
].copy()

print(f'Faixa de tamanho aplicada: {MINIMO_CHARS} – {MAXIMO_CHARS} chars')
print()
print('Registros apos filtro de tamanho:')
for lbl in [0, 1]:
    s = df_filtrado[df_filtrado['label'] == lbl]
    nome = 'fake' if lbl == 0 else 'real'
    print(f'  label={lbl} ({nome}): {len(s)} | media={s["tamanho_chars"].mean():.0f} | mediana={s["tamanho_chars"].median():.0f}')
print()

# Balancear por label (min das duas classes)
n_por_label = df_filtrado['label'].value_counts()
n_minimo = int(n_por_label.min())
print(f'Balanceamento: {n_minimo} por label (min das duas classes)')
print()

partes = []
for lbl in [0, 1]:
    parte = df_filtrado[df_filtrado['label'] == lbl].sample(n=n_minimo, random_state=42)
    partes.append(parte)

df_size_control = pd.concat(partes, ignore_index=True)
df_size_control = df_size_control[COLUNAS_CURATED].copy()

ARQUIVO_SIZE_CONTROL = CURATED_DIR / f'faketruebr_curated_size_control_{TS}.csv'
df_size_control.to_csv(ARQUIVO_SIZE_CONTROL, index=False, encoding='utf-8')

print('Apos balanceamento:')
for lbl in [0, 1]:
    s = df_size_control[df_size_control['label'] == lbl]
    nome = 'fake' if lbl == 0 else 'real'
    print(f'  label={lbl} ({nome}): {len(s)} | media={s["tamanho_chars"].mean():.0f} | mediana={s["tamanho_chars"].median():.0f}')
print()
print(f'Total size_control: {len(df_size_control)}')
print(f'Arquivo salvo: {ARQUIVO_SIZE_CONTROL.name}')
print(f'Caminho      : {ARQUIVO_SIZE_CONTROL}')

SECAO 6.2 — GERANDO faketruebr_curated_size_control
Faixa de tamanho aplicada: 300 – 3000 chars

Registros apos filtro de tamanho:
  label=0 (fake): 1519 | media=876 | mediana=709
  label=1 (real): 892 | media=1716 | mediana=1682

Balanceamento: 892 por label (min das duas classes)



Apos balanceamento:
  label=0 (fake): 892 | media=871 | mediana=702
  label=1 (real): 892 | media=1716 | mediana=1682

Total size_control: 1784
Arquivo salvo: faketruebr_curated_size_control_2026-05-24_20-01-57.csv
Caminho      : C:\Users\offan\Desktop\ml-checkai\dados\pipeline_datasets_academicos\curated\faketruebr\faketruebr_curated_size_control_2026-05-24_20-01-57.csv


## Seção 7 — Relatório final

In [9]:
AVISO = '!' * 65
print(SEP)
print('RELATORIO FINAL — curadoria_datasets_academicos (FakeTrueBR)')
print(SEP)
print()
print(f'[1] Arquivo raw usado      : {ARQUIVO_RAW.name}')
print()
print('[2] Contagens por etapa:')
print(f'  Total inicial (raw)      : {n_inicial}')
print(f'  Apos limpeza basica      : {n_apos_limpeza}')
print(f'    Removidos (sem texto)  : {n_sem_texto}')
print(f'    Removidos (sem label)  : {n_sem_label}')
print(f'  Apos deduplicacao        : {n_apos_dedup}')
print(f'    Removidos (texto dup)  : {n_removidos_texto}')
print(f'    Removidos (URL dup)    : {n_removidos_url}')
print(f'  Removidos (conflito)     : {n_conflitos_total}')
print(f'  Total curated_full       : {len(df_curated_full)}')
print(f'  Total size_control       : {len(df_size_control)}')
print()
print('[3] curated_full — distribuicao e tamanho por label:')
for lbl in [0, 1]:
    s = df_curated_full[df_curated_full['label'] == lbl]
    nome = 'fake' if lbl == 0 else 'real'
    print(f'  label={lbl} ({nome}): {len(s)} registros | media={s["tamanho_chars"].mean():.0f} | mediana={s["tamanho_chars"].median():.0f}')
print()
print('[4] curated_size_control — distribuicao e tamanho por label:')
for lbl in [0, 1]:
    s = df_size_control[df_size_control['label'] == lbl]
    nome = 'fake' if lbl == 0 else 'real'
    print(f'  label={lbl} ({nome}): {len(s)} registros | media={s["tamanho_chars"].mean():.0f} | mediana={s["tamanho_chars"].median():.0f}')
print()
print('[5] Vies de tamanho:')
print(f'  Ratio de medianas (full): {ratio_mediana:.2f}x')
print(f'  Vies relevante (>= 2.0x): {"SIM" if VIES_RELEVANTE else "NAO"}')
print()
print('[6] Arquivos salvos:')
print(f'  {ARQUIVO_FULL}')
print(f'  {ARQUIVO_SIZE_CONTROL}')
if ARQUIVO_CONFLITOS:
    print(f'  {ARQUIVO_CONFLITOS}  [conflitos]')
print()
print('[7] Recomendacoes:')
print('  curated_full        → pode ser usado na V3 full (treino com class_weight)')
print('  curated_size_control → usar como ablacao / controle metodologico na V3')
print()
print(AVISO)
print('NAO montar dataset_final_treino_v3 antes de:')
print('  1. Importar e curar Fake.Br Corpus')
print('  2. Verificar overlap cross-dataset (FakeTrueBR x V2)')
print('  3. Obter aprovacao do plano de montagem V3')
print(AVISO)
print(SEP)

RELATORIO FINAL — curadoria_datasets_academicos (FakeTrueBR)

[1] Arquivo raw usado      : faketruebr_raw_padronizado_2026-05-24_19-57-33.csv

[2] Contagens por etapa:
  Total inicial (raw)      : 3582
  Apos limpeza basica      : 3582
    Removidos (sem texto)  : 0
    Removidos (sem label)  : 0
  Apos deduplicacao        : 3095
    Removidos (texto dup)  : 400
    Removidos (URL dup)    : 87
  Removidos (conflito)     : 0
  Total curated_full       : 3095
  Total size_control       : 1784

[3] curated_full — distribuicao e tamanho por label:
  label=0 (fake): 1752 registros | media=861 | mediana=654
  label=1 (real): 1343 registros | media=3080 | mediana=2198

[4] curated_size_control — distribuicao e tamanho por label:
  label=0 (fake): 892 registros | media=871 | mediana=702
  label=1 (real): 892 registros | media=1716 | mediana=1682

[5] Vies de tamanho:
  Ratio de medianas (full): 3.36x
  Vies relevante (>= 2.0x): SIM

[6] Arquivos salvos:
  C:\Users\offan\Desktop\ml-checkai\dado

---

# Seção B — Curadoria do Fake.Br Corpus

**Dataset:** Fake.Br Corpus  
**Referência:** Monteiro et al., *Contributions to the Study of Fake News in Portuguese: New Corpus and Automatic Detection Results*, PROPOR 2018.

Aplica o mesmo pipeline de curadoria do FakeTrueBR ao Fake.Br Corpus.  
Saídas em `dados/pipeline_datasets_academicos/curated/fakebr/`.

**Constraints:**
- Arquivos FakeTrueBR não são modificados
- Nenhum arquivo existente é sobrescrito — timestamp em todos os outputs
- Nenhum modelo é treinado
- `dataset_final_treino_v3` não é criado aqui

In [10]:
SEP_B = '=' * 65

# --- Localizar raw mais recente do Fake.Br ---
RAW_FAKEBR_DIR_C  = PROJECT_ROOT / 'dados' / 'pipeline_datasets_academicos' / 'raw' / 'fakebr'
CURATED_FAKEBR_DIR = PROJECT_ROOT / 'dados' / 'pipeline_datasets_academicos' / 'curated' / 'fakebr'
CURATED_FAKEBR_DIR.mkdir(parents=True, exist_ok=True)

padronizados_fb = sorted(RAW_FAKEBR_DIR_C.glob('fakebr_raw_padronizado_*.csv'))
assert padronizados_fb, f'Nenhum fakebr_raw_padronizado_*.csv em {RAW_FAKEBR_DIR_C}'
ARQUIVO_RAW_FB = padronizados_fb[-1]

TS_FB_CUR = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')

print(SEP_B)
print('SECAO B1 — DIAGNOSTICO INICIAL — Fake.Br Corpus')
print(SEP_B)
print(f'Arquivo raw: {ARQUIVO_RAW_FB.name}')

df_raw_fb = pd.read_csv(ARQUIVO_RAW_FB, encoding='utf-8')
print(f'Shape      : {df_raw_fb.shape}')
print()

# Validar schema
cols_faltando_fb = [c for c in COLUNAS_SCHEMA if c not in df_raw_fb.columns]
if cols_faltando_fb:
    print(f'ATENCAO — Colunas faltando: {cols_faltando_fb}')
else:
    print('Schema CheckAI OK — todas as colunas obrigatorias presentes.')
print()

print('--- Distribuicao por label ---')
for lbl in [0, 1]:
    cnt = (df_raw_fb['label'] == lbl).sum()
    nome = 'fake' if lbl == 0 else 'real'
    print(f'  label={lbl} ({nome}): {cnt}')
print()

print('--- Tamanho por label ---')
for lbl in [0, 1]:
    s = df_raw_fb[df_raw_fb['label'] == lbl]['tamanho_chars']
    nome = 'fake' if lbl == 0 else 'real'
    print(f'label={lbl} ({nome}): media={s.mean():.0f} | mediana={s.median():.0f} | min={s.min()} | max={s.max()}')
print()

print('--- Percentis de tamanho por label ---')
for lbl in [0, 1]:
    s = df_raw_fb[df_raw_fb['label'] == lbl]['tamanho_chars']
    nome = 'fake' if lbl == 0 else 'real'
    print(f'label={lbl} ({nome}):')
    for p in [10, 25, 50, 75, 90, 95]:
        print(f'  P{p:2d}: {s.quantile(p/100):.0f}')
print()

print('--- Portal de origem (top 10) ---')
print(df_raw_fb['portal_origem'].value_counts().head(10).to_string())
print()

n_com_url_fb  = (df_raw_fb['url_origem'].fillna('').str.strip() != '').sum()
n_com_data_fb = (df_raw_fb['data_publicacao'].fillna('').astype(str).str.strip().replace('', np.nan).notna()).sum()
print(f'Registros com URL : {n_com_url_fb} / {len(df_raw_fb)}')
print(f'Registros com data: {n_com_data_fb} / {len(df_raw_fb)}')
print()

print('--- Exemplos curtos (< 200 chars) ---')
curtos_fb = df_raw_fb[df_raw_fb['tamanho_chars'] < 200]
if curtos_fb.empty:
    print('  (nenhum < 200 chars)')
else:
    for _, row in curtos_fb.head(3).iterrows():
        print(f"  [{row['label_detalhe']}] ({row['tamanho_chars']} chars): {str(row['texto_principal'])[:120]}")
print()
print('--- Exemplos longos (> 20000 chars) ---')
longos_fb = df_raw_fb[df_raw_fb['tamanho_chars'] > 20000]
if longos_fb.empty:
    print('  (nenhum > 20000 chars)')
else:
    for _, row in longos_fb.head(3).iterrows():
        print(f"  [{row['label_detalhe']}] ({row['tamanho_chars']} chars): {str(row['texto_principal'])[:120]}...")


SECAO B1 — DIAGNOSTICO INICIAL — Fake.Br Corpus
Arquivo raw: fakebr_raw_padronizado_2026-05-24_19-58-38.csv


Shape      : (7200, 13)

Schema CheckAI OK — todas as colunas obrigatorias presentes.

--- Distribuicao por label ---
  label=0 (fake): 3600
  label=1 (real): 3600

--- Tamanho por label ---
label=0 (fake): media=1124 | mediana=956 | min=45 | max=13280
label=1 (real): media=6673 | mediana=5581 | min=114 | max=46084

--- Percentis de tamanho por label ---
label=0 (fake):
  P10: 510
  P25: 696
  P50: 956
  P75: 1355
  P90: 1891
  P95: 2313
label=1 (real):
  P10: 2690
  P25: 3872
  P50: 5581
  P75: 8592
  P90: 12022
  P95: 13758

--- Portal de origem (top 10) ---
portal_origem
diariodobrasil.org        3333
G1 Globo                  2275
Estadão                   1202
afolhabrasil.com.br        173
Folha de S.Paulo            95
thejornalbrasil.com.br      66
DESCONHECIDO                26
CeticismoPolítico           16
topfivetv.com                7
Correio                      5

Registros com URL : 7174 / 7200
Registros com data: 5879 / 7200

--- Exemplos curtos (< 200 chars) ---
  [FA

In [11]:
SEP_B = '=' * 65

print(SEP_B)
print('SECAO B2 — LIMPEZA, DEDUPLICACAO E CONFLITOS — Fake.Br')
print(SEP_B)

df_clean_fb = df_raw_fb.copy()
n_inicial_fb = len(df_clean_fb)

# 1. Remover texto vazio/nulo
df_clean_fb['texto_principal'] = df_clean_fb['texto_principal'].astype(str)
mask_vazio_fb = (
    df_clean_fb['texto_principal'].isin(['', 'nan', 'None', 'NaN'])
    | df_clean_fb['texto_principal'].isna()
)
n_sem_texto_fb = int(mask_vazio_fb.sum())
df_clean_fb = df_clean_fb[~mask_vazio_fb].copy()

# 2. Remover sem label válido
mask_sem_label_fb = ~df_clean_fb['label'].isin([0, 1])
n_sem_label_fb = int(mask_sem_label_fb.sum())
df_clean_fb = df_clean_fb[~mask_sem_label_fb].copy()

# 3. Limpar HTML residual + normalizar espaços
import re as _re
df_clean_fb['texto_principal'] = (
    df_clean_fb['texto_principal']
    .str.replace(r'<[^>]+>', ' ', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

# 4. Recalcular tamanho_chars
df_clean_fb['tamanho_chars'] = df_clean_fb['texto_principal'].str.len()

# 5. Criar faixa_tamanho
def _faixa_fb(n):
    if n < 500: return 'curto'
    elif n < 2000: return 'medio'
    elif n < 5000: return 'longo'
    else: return 'muito_longo'

df_clean_fb['faixa_tamanho'] = df_clean_fb['tamanho_chars'].apply(_faixa_fb)
n_apos_limpeza_fb = len(df_clean_fb)

# 6. Criar texto_norm e deduplicar por texto
df_clean_fb['texto_norm'] = (
    df_clean_fb['texto_principal']
    .str.lower().str.strip()
    .str.replace(r'\s+', ' ', regex=True)
)
n_antes_dedup_fb = len(df_clean_fb)
n_dup_texto_fb = int(df_clean_fb.duplicated(subset=['texto_norm']).sum())
df_clean_fb = df_clean_fb.drop_duplicates(subset=['texto_norm'], keep='first').copy()
n_removidos_texto_fb = n_antes_dedup_fb - len(df_clean_fb)

# 7. Deduplicar por url_origem
mask_tem_url_fb = df_clean_fb['url_origem'].fillna('').str.strip() != ''
n_dup_url_fb = int(df_clean_fb[mask_tem_url_fb].duplicated(subset=['url_origem']).sum()) if mask_tem_url_fb.any() else 0
if mask_tem_url_fb.any() and n_dup_url_fb > 0:
    idx_url_ok_fb = df_clean_fb[mask_tem_url_fb].drop_duplicates(subset=['url_origem'], keep='first').index
    idx_sem_url_fb = df_clean_fb[~mask_tem_url_fb].index
    df_clean_fb = df_clean_fb.loc[sorted(idx_url_ok_fb.union(idx_sem_url_fb))].copy()
n_removidos_url_fb = n_antes_dedup_fb - n_removidos_texto_fb - len(df_clean_fb)
n_apos_dedup_fb = len(df_clean_fb)

# 8. Detecção de conflitos
dup_texto_conf_fb = (
    df_clean_fb.groupby('texto_norm')['label']
    .nunique().reset_index(name='n_labels')
)
textos_conflito_fb = set(dup_texto_conf_fb[dup_texto_conf_fb['n_labels'] > 1]['texto_norm'])

mask_tem_url_cur = df_clean_fb['url_origem'].fillna('').str.strip() != ''
if mask_tem_url_cur.any():
    dup_url_conf_fb = (
        df_clean_fb[mask_tem_url_cur].groupby('url_origem')['label']
        .nunique().reset_index(name='n_labels')
    )
    urls_conflito_fb = set(dup_url_conf_fb[dup_url_conf_fb['n_labels'] > 1]['url_origem'])
else:
    urls_conflito_fb = set()

mask_conflito_fb = (
    df_clean_fb['texto_norm'].isin(textos_conflito_fb)
    | df_clean_fb['url_origem'].isin(urls_conflito_fb)
)
df_conflitos_fb = df_clean_fb[mask_conflito_fb].copy()
df_sem_conflito_fb = df_clean_fb[~mask_conflito_fb].copy()
n_conflitos_fb = len(df_conflitos_fb)

ARQUIVO_CONFLITOS_FB = None
if n_conflitos_fb > 0:
    ARQUIVO_CONFLITOS_FB = CURATED_FAKEBR_DIR / f'fakebr_conflitos_{TS_FB_CUR}.csv'
    cols_conf_fb = [c for c in df_conflitos_fb.columns if c != 'texto_norm']
    df_conflitos_fb[cols_conf_fb].to_csv(ARQUIVO_CONFLITOS_FB, index=False, encoding='utf-8')

print(f'Registros iniciais         : {n_inicial_fb}')
print(f'  Removidos (sem texto)    : {n_sem_texto_fb}')
print(f'  Removidos (sem label)    : {n_sem_label_fb}')
print(f'Apos limpeza               : {n_apos_limpeza_fb}')
print(f'  Dup por texto_norm       : {n_dup_texto_fb} → removidos: {n_removidos_texto_fb}')
print(f'  Dup por url_origem       : {n_dup_url_fb} → removidos: {n_removidos_url_fb}')
print(f'Apos deduplicacao          : {n_apos_dedup_fb}')
print(f'Conflitos detectados       : {n_conflitos_fb}')
print(f'Registros limpos finais    : {len(df_sem_conflito_fb)}')
print()
print('--- Distribuicao por label apos curadoria ---')
for lbl in [0, 1]:
    cnt = (df_sem_conflito_fb['label'] == lbl).sum()
    nome = 'fake' if lbl == 0 else 'real'
    print(f'  label={lbl} ({nome}): {cnt}')
if n_conflitos_fb > 0 and ARQUIVO_CONFLITOS_FB:
    print(f'Conflitos salvos: {ARQUIVO_CONFLITOS_FB.name}')
else:
    print('Nenhum conflito de label detectado.')


SECAO B2 — LIMPEZA, DEDUPLICACAO E CONFLITOS — Fake.Br


Registros iniciais         : 7200
  Removidos (sem texto)    : 0
  Removidos (sem label)    : 0
Apos limpeza               : 7200
  Dup por texto_norm       : 1 → removidos: 1
  Dup por url_origem       : 17 → removidos: 17
Apos deduplicacao          : 7182
Conflitos detectados       : 0
Registros limpos finais    : 7182

--- Distribuicao por label apos curadoria ---
  label=0 (fake): 3600
  label=1 (real): 3582
Nenhum conflito de label detectado.


In [12]:
SEP_B = '=' * 65

print(SEP_B)
print('SECAO B3 — AUDITORIA DE VIES DE TAMANHO — Fake.Br')
print(SEP_B)

medias_fb   = {}
medianas_fb = {}
for lbl in [0, 1]:
    s = df_sem_conflito_fb[df_sem_conflito_fb['label'] == lbl]['tamanho_chars']
    medias_fb[lbl]   = s.mean()
    medianas_fb[lbl] = s.median()

print('--- Estatisticas de tamanho por label ---')
for lbl in [0, 1]:
    s = df_sem_conflito_fb[df_sem_conflito_fb['label'] == lbl]['tamanho_chars']
    nome = 'fake' if lbl == 0 else 'real'
    print(f'label={lbl} ({nome}): media={medias_fb[lbl]:.0f} | mediana={medianas_fb[lbl]:.0f} | min={s.min()} | max={s.max()}')
print()

ratio_mediana_fb = max(medianas_fb[0], medianas_fb[1]) / min(medianas_fb[0], medianas_fb[1])
ratio_media_fb   = max(medias_fb[0], medias_fb[1]) / min(medias_fb[0], medias_fb[1])
print(f'Ratio medianas (real/fake): {ratio_mediana_fb:.2f}x')
print(f'Ratio medias   (real/fake): {ratio_media_fb:.2f}x')
print()

print('--- Percentis de tamanho por label ---')
for lbl in [0, 1]:
    s = df_sem_conflito_fb[df_sem_conflito_fb['label'] == lbl]['tamanho_chars']
    nome = 'fake' if lbl == 0 else 'real'
    print(f'label={lbl} ({nome}):')
    for p in [10, 25, 50, 75, 90, 95]:
        print(f'  P{p:2d}: {s.quantile(p/100):.0f}')
print()

print('--- Tabela cruzada: label x faixa_tamanho ---')
ordem_faixas_fb = ['curto', 'medio', 'longo', 'muito_longo']
tab_fb = pd.crosstab(df_sem_conflito_fb['faixa_tamanho'], df_sem_conflito_fb['label'], margins=True)
tab_fb.columns = ['label=0 (fake)', 'label=1 (real)', 'Total']
tab_fb.index.name = 'faixa_tamanho'
idx_ex_fb = [f for f in ordem_faixas_fb if f in tab_fb.index] + ['All']
tab_fb = tab_fb.reindex(idx_ex_fb)
print(tab_fb.to_string())
print()

print('--- Portal de origem (top 10) ---')
print(df_sem_conflito_fb['portal_origem'].value_counts().head(10).to_string())
print()

VIES_RELEVANTE_FB = ratio_mediana_fb >= 2.0
print('--- Diagnostico de vies ---')
if VIES_RELEVANTE_FB:
    print(f'[VIES RELEVANTE] Ratio de medianas = {ratio_mediana_fb:.2f}x (>= 2.0 — limite metodologico).')
    print(f'  Mediana label=1 ({medianas_fb[1]:.0f}) e {ratio_mediana_fb:.2f}x maior que label=0 ({medianas_fb[0]:.0f}).')
    print('  Recomendacao: usar curated_size_control como ablacao.')
else:
    print(f'[VIES MODERADO] Ratio = {ratio_mediana_fb:.2f}x (< 2.0).')
print()
print('--- Comparacao com FakeTrueBR ---')
print(f'  FakeTrueBR ratio_mediana: 3.36x')
print(f'  Fake.Br    ratio_mediana: {ratio_mediana_fb:.2f}x')
if ratio_mediana_fb > 3.36:
    print('  -> Fake.Br tem vies de tamanho MAIOR que FakeTrueBR.')
elif ratio_mediana_fb < 3.36:
    print('  -> Fake.Br tem vies de tamanho MENOR que FakeTrueBR.')
else:
    print('  -> Vies similar ao FakeTrueBR.')


SECAO B3 — AUDITORIA DE VIES DE TAMANHO — Fake.Br
--- Estatisticas de tamanho por label ---
label=0 (fake): media=1118 | mediana=952 | min=45 | max=13250
label=1 (real): media=6671 | mediana=5580 | min=113 | max=46083

Ratio medianas (real/fake): 5.86x
Ratio medias   (real/fake): 5.97x

--- Percentis de tamanho por label ---
label=0 (fake):
  P10: 505
  P25: 692
  P50: 952
  P75: 1348
  P90: 1883
  P95: 2305
label=1 (real):
  P10: 2690
  P25: 3870
  P50: 5580
  P75: 8587
  P90: 12022
  P95: 13754

--- Tabela cruzada: label x faixa_tamanho ---
               label=0 (fake)  label=1 (real)  Total
faixa_tamanho                                       
curto                     346               2    348
medio                    2965             161   3126
longo                     275            1350   1625
muito_longo                14            2069   2083
All                      3600            3582   7182

--- Portal de origem (top 10) ---
portal_origem
diariodobrasil.org        3333


In [13]:
SEP_B = '=' * 65

print(SEP_B)
print('SECAO B4 — GERANDO fakebr_curated_full')
print(SEP_B)

COLUNAS_CURATED_FB = COLUNAS_SCHEMA + ['faixa_tamanho']
df_curated_full_fb = df_sem_conflito_fb[COLUNAS_CURATED_FB].copy().reset_index(drop=True)

ARQUIVO_FULL_FB = CURATED_FAKEBR_DIR / f'fakebr_curated_full_{TS_FB_CUR}.csv'
df_curated_full_fb.to_csv(ARQUIVO_FULL_FB, index=False, encoding='utf-8')

print(f'Total registros : {len(df_curated_full_fb)}')
for lbl in [0, 1]:
    s = df_curated_full_fb[df_curated_full_fb['label'] == lbl]
    nome = 'fake' if lbl == 0 else 'real'
    print(f'  label={lbl} ({nome}): {len(s)} | media={s["tamanho_chars"].mean():.0f} | mediana={s["tamanho_chars"].median():.0f}')
print()
print('Distribuicao por faixa_tamanho:')
ordem_fb = ['curto', 'medio', 'longo', 'muito_longo']
tab_full_fb = pd.crosstab(df_curated_full_fb['faixa_tamanho'], df_curated_full_fb['label'], margins=True)
tab_full_fb.columns = ['label=0 (fake)', 'label=1 (real)', 'Total']
idx_fb_ex = [f for f in ordem_fb if f in tab_full_fb.index] + ['All']
tab_full_fb = tab_full_fb.reindex(idx_fb_ex)
print(tab_full_fb.to_string())
print()
print(f'Arquivo salvo: {ARQUIVO_FULL_FB.name}')
print(f'Caminho      : {ARQUIVO_FULL_FB}')


SECAO B4 — GERANDO fakebr_curated_full


Total registros : 7182
  label=0 (fake): 3600 | media=1118 | mediana=952
  label=1 (real): 3582 | media=6671 | mediana=5580

Distribuicao por faixa_tamanho:
               label=0 (fake)  label=1 (real)  Total
faixa_tamanho                                       
curto                     346               2    348
medio                    2965             161   3126
longo                     275            1350   1625
muito_longo                14            2069   2083
All                      3600            3582   7182

Arquivo salvo: fakebr_curated_full_2026-05-24_20-01-58.csv
Caminho      : C:\Users\offan\Desktop\ml-checkai\dados\pipeline_datasets_academicos\curated\fakebr\fakebr_curated_full_2026-05-24_20-01-58.csv


In [14]:
SEP_B = '=' * 65

print(SEP_B)
print('SECAO B5 — GERANDO fakebr_curated_size_control')
print(SEP_B)

MINIMO_CHARS_FB = 300
MAXIMO_CHARS_FB = 3000

df_filtrado_fb = df_sem_conflito_fb[
    (df_sem_conflito_fb['tamanho_chars'] >= MINIMO_CHARS_FB) &
    (df_sem_conflito_fb['tamanho_chars'] <= MAXIMO_CHARS_FB)
].copy()

print(f'Faixa de tamanho: {MINIMO_CHARS_FB} – {MAXIMO_CHARS_FB} chars')
print()
print('Registros apos filtro de tamanho:')
for lbl in [0, 1]:
    s = df_filtrado_fb[df_filtrado_fb['label'] == lbl]
    nome = 'fake' if lbl == 0 else 'real'
    if len(s) == 0:
        print(f'  label={lbl} ({nome}): 0 registros')
    else:
        print(f'  label={lbl} ({nome}): {len(s)} | media={s["tamanho_chars"].mean():.0f} | mediana={s["tamanho_chars"].median():.0f}')
print()

n_por_label_fb = df_filtrado_fb['label'].value_counts()
n_min_fb = int(n_por_label_fb.min())

if n_min_fb == 0:
    print('AVISO: Faixa 300-3000 removeu todas as entradas de uma classe.')
    print('Usando curated_full como fallback.')
    df_size_control_fb = df_curated_full_fb.copy()
else:
    print(f'Balanceamento: {n_min_fb} por label (min das duas classes)')
    partes_fb = []
    for lbl in [0, 1]:
        parte = df_filtrado_fb[df_filtrado_fb['label'] == lbl].sample(n=n_min_fb, random_state=42)
        partes_fb.append(parte)
    df_size_control_fb = pd.concat(partes_fb, ignore_index=True)
    df_size_control_fb = df_size_control_fb[COLUNAS_CURATED_FB].copy()

ARQUIVO_SIZE_CONTROL_FB = CURATED_FAKEBR_DIR / f'fakebr_curated_size_control_{TS_FB_CUR}.csv'
df_size_control_fb.to_csv(ARQUIVO_SIZE_CONTROL_FB, index=False, encoding='utf-8')

print()
print('Apos balanceamento:')
for lbl in [0, 1]:
    s = df_size_control_fb[df_size_control_fb['label'] == lbl]
    nome = 'fake' if lbl == 0 else 'real'
    print(f'  label={lbl} ({nome}): {len(s)} | media={s["tamanho_chars"].mean():.0f} | mediana={s["tamanho_chars"].median():.0f}')
print(f'  Ratio medianas apos controle: {df_size_control_fb[df_size_control_fb.label==1]["tamanho_chars"].median() / df_size_control_fb[df_size_control_fb.label==0]["tamanho_chars"].median():.2f}x')
print()
print(f'Total size_control: {len(df_size_control_fb)}')
print(f'Arquivo salvo: {ARQUIVO_SIZE_CONTROL_FB.name}')
print(f'Caminho      : {ARQUIVO_SIZE_CONTROL_FB}')


SECAO B5 — GERANDO fakebr_curated_size_control
Faixa de tamanho: 300 – 3000 chars

Registros apos filtro de tamanho:
  label=0 (fake): 3419 | media=1067 | mediana=954
  label=1 (real): 464 | media=2168 | mediana=2312

Balanceamento: 464 por label (min das duas classes)

Apos balanceamento:
  label=0 (fake): 464 | media=1091 | mediana=960
  label=1 (real): 464 | media=2168 | mediana=2312
  Ratio medianas apos controle: 2.41x

Total size_control: 928
Arquivo salvo: fakebr_curated_size_control_2026-05-24_20-01-58.csv
Caminho      : C:\Users\offan\Desktop\ml-checkai\dados\pipeline_datasets_academicos\curated\fakebr\fakebr_curated_size_control_2026-05-24_20-01-58.csv


In [15]:
SEP_B = '=' * 65
AVISO_B = '!' * 65
print(SEP_B)
print('RELATORIO FINAL — curadoria Fake.Br + Comparacao com FakeTrueBR')
print(SEP_B)
print()

# --- Fake.Br Corpus ---
print('[Fake.Br Corpus — resumo]')
print(f'  Arquivo raw              : {ARQUIVO_RAW_FB.name}')
print(f'  Total inicial            : {n_inicial_fb}')
print(f'  Apos limpeza             : {n_apos_limpeza_fb}')
print(f'  Apos dedup               : {n_apos_dedup_fb}')
print(f'    Removidos (texto dup)  : {n_removidos_texto_fb}')
print(f'    Removidos (URL dup)    : {n_removidos_url_fb}')
print(f'  Conflitos detectados     : {n_conflitos_fb}')
print(f'  curated_full             : {len(df_curated_full_fb)} registros')
print(f'  curated_size_control     : {len(df_size_control_fb)} registros')
print(f'  Vies de tamanho          : {ratio_mediana_fb:.2f}x ({"RELEVANTE" if VIES_RELEVANTE_FB else "MODERADO"})')
print()

# --- Tabela comparativa ---
fb_fake_full = int((df_curated_full_fb['label'] == 0).sum())
fb_real_full = int((df_curated_full_fb['label'] == 1).sum())
fb_med_fake  = float(df_curated_full_fb[df_curated_full_fb['label']==0]['tamanho_chars'].median())
fb_med_real  = float(df_curated_full_fb[df_curated_full_fb['label']==1]['tamanho_chars'].median())

ftbr_fake_full, ftbr_real_full = 1752, 1343
ftbr_med_fake, ftbr_med_real   = 654.0, 2198.0
ftbr_size_total, ftbr_ratio    = 1784, 3.36

print('[COMPARACAO: FakeTrueBR vs Fake.Br Corpus]')
print()
print(f'{"Metrica":<40} {"FakeTrueBR":>12} {"Fake.Br":>10}')
print('-' * 65)
print(f'{"curated_full — total":<40} {"3095":>12} {len(df_curated_full_fb):>10}')
print(f'{"curated_full — label=0 (fake)":<40} {ftbr_fake_full:>12} {fb_fake_full:>10}')
print(f'{"curated_full — label=1 (real)":<40} {ftbr_real_full:>12} {fb_real_full:>10}')
print(f'{"curated_full — mediana fake (chars)":<40} {ftbr_med_fake:>12.0f} {fb_med_fake:>10.0f}')
print(f'{"curated_full — mediana real (chars)":<40} {ftbr_med_real:>12.0f} {fb_med_real:>10.0f}')
print(f'{"curated_full — ratio mediana (real/fake)":<40} {ftbr_ratio:>11.2f}x {ratio_mediana_fb:>9.2f}x')
print(f'{"size_control — total":<40} {"1784":>12} {len(df_size_control_fb):>10}')
print()

# --- Riscos metodológicos ---
print('[RISCOS METODOLOGICOS]')
print()
print('FakeTrueBR:')
print('  - Vies estrutural de tamanho (3.36x) — relevante')
print('  - fake = Boatos.org; real = G1/Folha — vies de portal unico')
print('  - 487 duplicatas internas removidas (13.6%)')
print('  - Desbalanceamento apos dedup: 1752 fake / 1343 real')
print()
print('Fake.Br Corpus:')
if VIES_RELEVANTE_FB:
    print(f'  - Vies estrutural de tamanho ({ratio_mediana_fb:.2f}x) — MUITO RELEVANTE')
else:
    print(f'  - Vies de tamanho: {ratio_mediana_fb:.2f}x (moderado)')
print('  - fake de fake-news sites; real de G1/Estadao — vies de portal')
print('  - Dataset 2017-2018 — viés temporal concentrado')
print('  - URLs e datas disponíveis via metadados')
print('  - Volume grande (7200 raw) mas com alto viés size')
print()

print('[RECOMENDACOES]')
print('  FakeTrueBR curated_full   → V3 full (class_weight obrigatorio)')
print('  FakeTrueBR size_control   → ablacao size-controlled V3')
print('  Fake.Br curated_full      → V3 full (class_weight obrigatorio)')
print('  Fake.Br size_control      → ablacao size-controlled V3')
print()
print('[ARQUIVOS SALVOS]')
print(f'  {ARQUIVO_FULL_FB}')
print(f'  {ARQUIVO_SIZE_CONTROL_FB}')
if ARQUIVO_CONFLITOS_FB:
    print(f'  {ARQUIVO_CONFLITOS_FB}  [conflitos]')
print()
print(AVISO_B)
print('NAO montar dataset_final_treino_v3 antes de:')
print('  1. Verificar overlap cross-dataset (FakeTrueBR x Fake.Br x V2)')
print('  2. Obter aprovacao do plano de montagem V3')
print(AVISO_B)
print(SEP_B)


RELATORIO FINAL — curadoria Fake.Br + Comparacao com FakeTrueBR

[Fake.Br Corpus — resumo]
  Arquivo raw              : fakebr_raw_padronizado_2026-05-24_19-58-38.csv
  Total inicial            : 7200
  Apos limpeza             : 7200
  Apos dedup               : 7182
    Removidos (texto dup)  : 1
    Removidos (URL dup)    : 17
  Conflitos detectados     : 0
  curated_full             : 7182 registros
  curated_size_control     : 928 registros
  Vies de tamanho          : 5.86x (RELEVANTE)

[COMPARACAO: FakeTrueBR vs Fake.Br Corpus]

Metrica                                    FakeTrueBR    Fake.Br
-----------------------------------------------------------------
curated_full — total                             3095       7182
curated_full — label=0 (fake)                    1752       3600
curated_full — label=1 (real)                    1343       3582
curated_full — mediana fake (chars)               654        952
curated_full — mediana real (chars)              2198       5580
cu


---

## Seção C — Fake.Br Corpus: Normalização de Texto para Treino

**Objetivo:** criar versão intermediária com `texto_principal_modelo` truncado  
para reduzir o viés de tamanho entre classes (ratio ≈ 5.86×) sem descartar registros.

**Estratégia:** testar limites N ∈ {1500, 2000, 2500} e escolher o maior N que traz ratio < 2.0.

**Saída:** `fakebr_curated_text_normalized_YYYY-MM-DD_HH-MM-SS.csv`

**Constraint:** nenhum arquivo existente é modificado — `texto_principal` original preservado para auditoria.


In [ ]:

from pathlib import Path
import pandas as pd
from datetime import datetime

SEP_C = '=' * 65

# --- PROJECT_ROOT robusto (funciona rodando de src/ ou da raiz) ---
def _find_project_root_c() -> Path:
    cwd = Path.cwd()
    if (cwd / 'dados').is_dir():
        return cwd
    if (cwd.parent / 'dados').is_dir():
        return cwd.parent
    raise FileNotFoundError(
        f'Pasta dados nao encontrada em {cwd}. '
        'Execute a partir da raiz do projeto ou de src/.'
    )

_PR_C = _find_project_root_c()
_CURATED_FAKEBR_C = _PR_C / 'dados' / 'pipeline_datasets_academicos' / 'curated' / 'fakebr'

# --- auto-localizar fakebr_curated_full mais recente ---
_full_files_c = sorted(_CURATED_FAKEBR_C.glob('fakebr_curated_full_*.csv'))
assert _full_files_c, f'Nenhum fakebr_curated_full_*.csv em {_CURATED_FAKEBR_C}'
_ARQUIVO_FULL_C = _full_files_c[-1]

print(f'PROJECT_ROOT  : {_PR_C}')
print(f'Entrada       : {_ARQUIVO_FULL_C.name}')

df_full_c = pd.read_csv(_ARQUIVO_FULL_C, dtype={'id_registro': str})
print(f'Registros     : {len(df_full_c):,}')
print()

print('--- Distribuicao por label ---')
for _lbl in [0, 1]:
    _cnt = (df_full_c['label'] == _lbl).sum()
    _nome = 'fake' if _lbl == 0 else 'real'
    print(f'  label={_lbl} ({_nome}): {_cnt:,}')
print()

print('--- Tamanho original (baseline) ---')
for _lbl in [0, 1]:
    _s = df_full_c[df_full_c['label'] == _lbl]['tamanho_chars']
    _nome = 'fake' if _lbl == 0 else 'real'
    print(f'  label={_lbl} ({_nome}): media={_s.mean():.0f} | mediana={_s.median():.0f} | max={_s.max()}')
_ratio_orig_c = (df_full_c[df_full_c['label']==1]['tamanho_chars'].median() /
                 df_full_c[df_full_c['label']==0]['tamanho_chars'].median())
print(f'  Ratio original (real/fake): {_ratio_orig_c:.2f}x')


In [ ]:

print(SEP_C)
print('SECAO C — DIAGNOSTICO: Testando N = 1500, 2000, 2500')
print(SEP_C)
print(f'Baseline (sem truncamento): ratio={_ratio_orig_c:.2f}x | mediana fake={df_full_c[df_full_c.label==0]["tamanho_chars"].median():.0f} | mediana real={df_full_c[df_full_c.label==1]["tamanho_chars"].median():.0f}')
print()

_VALORES_N = [1500, 2000, 2500]
_resultados_n = {}

for _N in _VALORES_N:
    _df_t = df_full_c.copy()
    _df_t['_modelo_chars'] = _df_t['texto_principal'].str[:_N].str.len()
    _df_t['_truncado'] = _df_t['tamanho_chars'] > _N

    _med_f = _df_t[_df_t['label']==0]['_modelo_chars'].median()
    _med_r = _df_t[_df_t['label']==1]['_modelo_chars'].median()
    _ratio = _med_r / _med_f

    _nf = int(_df_t[_df_t['label']==0]['_truncado'].sum())
    _nr = int(_df_t[_df_t['label']==1]['_truncado'].sum())
    _pf = _nf / len(_df_t[_df_t['label']==0]) * 100
    _pr = _nr / len(_df_t[_df_t['label']==1]) * 100

    _resultados_n[_N] = {
        'ratio': _ratio, 'med_fake': _med_f, 'med_real': _med_r,
        'n_trunc_fake': _nf, 'n_trunc_real': _nr,
        'pct_trunc_fake': _pf, 'pct_trunc_real': _pr,
    }

    print(f'  N={_N:,}')
    print(f'    mediana fake : {_med_f:.0f} | mediana real: {_med_r:.0f} | ratio: {_ratio:.2f}x')
    print(f'    trunc fake   : {_nf:,} ({_pf:.1f}%) | trunc real: {_nr:,} ({_pr:.1f}%)')
    print()

# --- escolha automática de N ---
# Maior N com ratio < 2.0; se nenhum, o que tiver menor ratio
_N_ESCOLHIDO = None
for _N in sorted(_VALORES_N, reverse=True):
    if _resultados_n[_N]['ratio'] < 2.0:
        _N_ESCOLHIDO = _N
        break
if _N_ESCOLHIDO is None:
    _N_ESCOLHIDO = min(_VALORES_N, key=lambda n: _resultados_n[n]['ratio'])
    print(f'Nenhum N trouxe ratio < 2.0.')
    print(f'Escolhido N={_N_ESCOLHIDO} (menor ratio: {_resultados_n[_N_ESCOLHIDO]["ratio"]:.2f}x)')
else:
    print(f'Escolhido: N={_N_ESCOLHIDO:,} (maior N com ratio < 2.0: {_resultados_n[_N_ESCOLHIDO]["ratio"]:.2f}x)')


In [ ]:

print(SEP_C)
print(f'SECAO C — APLICANDO N={_N_ESCOLHIDO:,} E SALVANDO')
print(SEP_C)

_TS_NORM = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
_SAIDA_NORM = _CURATED_FAKEBR_C / f'fakebr_curated_text_normalized_{_TS_NORM}.csv'

def _faixa_modelo(n):
    if n < 500: return 'curto'
    elif n < 2000: return 'medio'
    elif n < 5000: return 'longo'
    else: return 'muito_longo'

df_norm_c = df_full_c.copy()
df_norm_c['tamanho_chars_original'] = df_norm_c['tamanho_chars']
df_norm_c['texto_principal_modelo'] = df_norm_c['texto_principal'].str[:_N_ESCOLHIDO]
df_norm_c['tamanho_chars_modelo'] = df_norm_c['texto_principal_modelo'].str.len()
df_norm_c['texto_truncado'] = df_norm_c['tamanho_chars_original'] > _N_ESCOLHIDO
df_norm_c['faixa_tamanho_modelo'] = df_norm_c['tamanho_chars_modelo'].apply(_faixa_modelo)
df_norm_c['limite_truncamento'] = _N_ESCOLHIDO

# Reordenar: colunas originais primeiro, novas ao final
_COLS_NOVAS_C = ['tamanho_chars_original', 'texto_principal_modelo', 'tamanho_chars_modelo',
                 'faixa_tamanho_modelo', 'texto_truncado', 'limite_truncamento']
_cols_base_c = [c for c in df_norm_c.columns if c not in _COLS_NOVAS_C]
df_norm_c = df_norm_c[_cols_base_c + _COLS_NOVAS_C]

df_norm_c.to_csv(_SAIDA_NORM, index=False, encoding='utf-8')
print(f'Arquivo salvo : {_SAIDA_NORM.name}')
print(f'Registros     : {len(df_norm_c):,}')
print(f'Colunas       : {list(df_norm_c.columns)}')
print()
print('Amostra das novas colunas (3 registros label=1):')
_sample_cols_c = ['id_registro', 'label', 'tamanho_chars_original', 'tamanho_chars_modelo', 'texto_truncado', 'faixa_tamanho_modelo']
print(df_norm_c[df_norm_c['label']==1][_sample_cols_c].head(3).to_string(index=False))


In [ ]:

print(SEP_C)
print('RELATORIO FINAL — Fake.Br: Normalizacao de Texto para Treino')
print(SEP_C)

_df_fake_c = df_norm_c[df_norm_c['label'] == 0]
_df_real_c = df_norm_c[df_norm_c['label'] == 1]

print(f'\n[Parametros]')
print(f'  Entrada     : {_ARQUIVO_FULL_C.name}')
print(f'  Saida       : {_SAIDA_NORM.name}')
print(f'  N escolhido : {_N_ESCOLHIDO:,} chars')

print(f'\n[Volume]')
print(f'  Total de registros : {len(df_norm_c):,}')
print(f'  label=0 (fake)     : {len(_df_fake_c):,}')
print(f'  label=1 (real)     : {len(_df_real_c):,}')

print(f'\n[Estatisticas — texto_principal ORIGINAL]')
print(f'  fake — media: {_df_fake_c["tamanho_chars_original"].mean():.0f} | mediana: {_df_fake_c["tamanho_chars_original"].median():.0f}')
print(f'  real — media: {_df_real_c["tamanho_chars_original"].mean():.0f} | mediana: {_df_real_c["tamanho_chars_original"].median():.0f}')
_ratio_antes_c = _df_real_c['tamanho_chars_original'].median() / _df_fake_c['tamanho_chars_original'].median()
print(f'  ratio (real/fake) : {_ratio_antes_c:.2f}x')

print(f'\n[Estatisticas — texto_principal_modelo (N={_N_ESCOLHIDO:,})]')
print(f'  fake — media: {_df_fake_c["tamanho_chars_modelo"].mean():.0f} | mediana: {_df_fake_c["tamanho_chars_modelo"].median():.0f}')
print(f'  real — media: {_df_real_c["tamanho_chars_modelo"].mean():.0f} | mediana: {_df_real_c["tamanho_chars_modelo"].median():.0f}')
_ratio_depois_c = _df_real_c['tamanho_chars_modelo'].median() / _df_fake_c['tamanho_chars_modelo'].median()
print(f'  ratio (real/fake) : {_ratio_depois_c:.2f}x')
print(f'  Reducao do ratio  : {_ratio_antes_c:.2f}x → {_ratio_depois_c:.2f}x ({(1-_ratio_depois_c/_ratio_antes_c)*100:.1f}% de reducao)')

print(f'\n[Truncamento]')
_total_trunc = int(df_norm_c['texto_truncado'].sum())
print(f'  Total truncado : {_total_trunc:,} de {len(df_norm_c):,} ({_total_trunc/len(df_norm_c)*100:.1f}%)')
_trunc_fake = int(_df_fake_c['texto_truncado'].sum())
_trunc_real = int(_df_real_c['texto_truncado'].sum())
print(f'  fake : {_trunc_fake:,} de {len(_df_fake_c):,} ({_trunc_fake/len(_df_fake_c)*100:.1f}%)')
print(f'  real : {_trunc_real:,} de {len(_df_real_c):,} ({_trunc_real/len(_df_real_c)*100:.1f}%)')

print(f'\n[Faixa de tamanho — modelo]')
for _lbl, _nome, _sub in [(0, 'fake', _df_fake_c), (1, 'real', _df_real_c)]:
    _dist = _sub['faixa_tamanho_modelo'].value_counts()
    print(f'  label={_lbl} ({_nome}): ' + ' | '.join(f'{k}={_dist.get(k, 0)}' for k in ['curto', 'medio', 'longo', 'muito_longo']))

print(f'\n[Comparacao entre versoes do Fake.Br]')
_sc_files_c = sorted(_CURATED_FAKEBR_C.glob('fakebr_curated_size_control_*.csv'))
if _sc_files_c:
    _df_sc_c = pd.read_csv(_sc_files_c[-1], dtype={'id_registro': str})
    _ratio_sc = (_df_sc_c[_df_sc_c['label']==1]['tamanho_chars'].median() /
                 _df_sc_c[_df_sc_c['label']==0]['tamanho_chars'].median())
    print(f'  {"Versao":<35} {"Registros":>10}  {"Ratio":>8}')
    print(f'  {"-"*57}')
    print(f'  {"fakebr_curated_full":<35} {len(df_full_c):>10,}  {_ratio_antes_c:>7.2f}x')
    print(f'  {"fakebr_curated_size_control":<35} {len(_df_sc_c):>10,}  {_ratio_sc:>7.2f}x')
    print(f'  {"fakebr_curated_text_normalized":<35} {len(df_norm_c):>10,}  {_ratio_depois_c:>7.2f}x')
else:
    print(f'  fakebr_curated_full            : {len(df_full_c):,} registros | ratio {_ratio_antes_c:.2f}x')
    print(f'  fakebr_curated_text_normalized : {len(df_norm_c):,} registros | ratio {_ratio_depois_c:.2f}x')

print(f'\n[Recomendacao para V3]')
if _ratio_depois_c < 2.0:
    print(f'  RECOMENDADO: text_normalized (N={_N_ESCOLHIDO:,})')
    print(f'  Motivo: ratio {_ratio_depois_c:.2f}x < 2.0 (abaixo do limiar de vies relevante)')
    if _sc_files_c:
        _ganho = len(df_norm_c) - len(_df_sc_c)
        print(f'  Volume: {len(df_norm_c):,} registros (+{_ganho:,} vs size_control)')
    print(f'  Coluna de treino   : texto_principal_modelo')
    print(f'  Coluna de auditoria: texto_principal (original preservado)')
elif _ratio_depois_c < _ratio_antes_c * 0.6:
    print(f'  USAR COM CAUTELA: text_normalized (ratio {_ratio_depois_c:.2f}x, reducao de {(1-_ratio_depois_c/_ratio_antes_c)*100:.0f}%)')
    if _sc_files_c:
        print(f'  Alternativa: size_control ({len(_df_sc_c):,} registros) se ratio < 2.0 for obrigatorio')
else:
    print(f'  NAO RECOMENDADO como substituto do size_control (ratio {_ratio_depois_c:.2f}x ainda alto)')
    if _sc_files_c:
        print(f'  Preferir: size_control ({len(_df_sc_c):,} registros, ratio {_ratio_sc:.2f}x)')

print()
print('!' * 65)
print('AVISO: texto_principal_modelo e EXCLUSIVO para treino.')
print('Nao substituir texto_principal — original preservado para auditoria.')
print('!' * 65)
print(SEP_C)
